# Urdu ↔ English Neural Machine Translation
### End-to-End Deep Learning Project — Google Colab

**Project objective:** Build a small NLP translation system that translates:
- Urdu → English
- English → Urdu

This notebook follows the assignment phases: online dataset preparation, pretrained Hugging Face model selection, fine-tuning, evaluation, saving, Hugging Face upload, GitHub/Streamlit project-file generation, and deployment instructions.

**Assignment basis:** The supplied assignment requires a practical problem, suitable dataset, preprocessing, a pretrained Hugging Face model, fine-tuning, saving/hosting the model, GitHub source code, and a Streamlit deployment.

## 1. Project information

**Title:** Urdu ↔ English Neural Machine Translation using Transfer Learning

**Problem statement:**  
Many users need to translate between Urdu and English. This project fine-tunes pretrained sequence-to-sequence translation models on an online parallel Urdu-English dataset and provides a simple web application for both translation directions.

**Online dataset:** `Helsinki-NLP/opus-100`, configuration `en-ur`. OPUS-100 is a multilingual parallel translation corpus with English on one side of each pair.

**Pretrained models:**
- Urdu → English: `Helsinki-NLP/opus-mt-ur-en`
- English → Urdu: `Helsinki-NLP/opus-mt-en-ur`

Both are Hugging Face translation models specifically trained for the Urdu-English direction.

> **Important:** For a faster Colab demonstration, the default notebook uses a small training subset. You can increase `MAX_TRAIN_SAMPLES` after confirming that your GPU runtime works.

In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================
!pip -q install -U "transformers==4.46.3" "datasets==3.1.0" "evaluate==0.4.3" \
    "sentencepiece==0.2.0" "sacrebleu==2.4.3" "accelerate==1.0.1" \
    "huggingface_hub>=0.26.0" "streamlit==1.39.0"

print("Installation complete.")

In [ ]:
# ============================================================
# 3. Check the Colab GPU
# ============================================================
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. In Colab, use Runtime > Change runtime type > T4 GPU.")


In [ ]:
# ============================================================
# 4. Project configuration
# ============================================================
SEED = 42

# Small settings for a classroom/Colab project.
# Increase to 10000-30000 if you want stronger fine-tuning and have enough GPU time.
MAX_TRAIN_SAMPLES = 6000
MAX_VALID_SAMPLES = 500
MAX_TEST_SAMPLES = 500

NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

UR_EN_BASE = "Helsinki-NLP/opus-mt-ur-en"
EN_UR_BASE = "Helsinki-NLP/opus-mt-en-ur"

print("Configuration loaded.")


In [ ]:
# ============================================================
# 5. Load the online Urdu-English dataset
# ============================================================
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-ur")

print(dataset)
print("Example:")
print(dataset["train"][0])


In [ ]:
# ============================================================
# 6. Explore and clean the dataset
# ============================================================
import pandas as pd

print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

def valid_pair(example):
    en = str(example["translation"].get("en", "")).strip()
    ur = str(example["translation"].get("ur", "")).strip()
    return len(en) > 0 and len(ur) > 0

cleaned = dataset.filter(valid_pair)

# Shuffle first, then select a manageable subset.
train_ds = cleaned["train"].shuffle(seed=SEED).select(
    range(min(MAX_TRAIN_SAMPLES, len(cleaned["train"])))
)
valid_ds = cleaned["validation"].shuffle(seed=SEED).select(
    range(min(MAX_VALID_SAMPLES, len(cleaned["validation"])))
)
test_ds = cleaned["test"].shuffle(seed=SEED).select(
    range(min(MAX_TEST_SAMPLES, len(cleaned["test"])))
)

print("Selected training examples:", len(train_ds))
print("Selected validation examples:", len(valid_ds))
print("Selected test examples:", len(test_ds))

sample_df = pd.DataFrame([
    {
        "English": x["translation"]["en"],
        "Urdu": x["translation"]["ur"]
    }
    for x in train_ds.select(range(min(8, len(train_ds))))
])
display(sample_df)


## 7. Fine-tuning strategy

We use **transfer learning**: start from a pretrained translation model and fine-tune it on the Urdu-English parallel data.

The two directions use the corresponding pretrained Marian models. Hugging Face documents direct loading with `AutoTokenizer` and `AutoModelForSeq2SeqLM` for these models.

In [ ]:
# ============================================================
# 8. Imports and reusable fine-tuning function
# ============================================================
import os
import numpy as np
import evaluate
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

set_seed(SEED)
bleu_metric = evaluate.load("sacrebleu")

def prepare_direction(dataset_dict, model_id, source_lang, target_lang):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    def preprocess(batch):
        sources = [str(x[source_lang]) for x in batch["translation"]]
        targets = [str(x[target_lang]) for x in batch["translation"]]

        model_inputs = tokenizer(
            sources,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True
        )

        labels = tokenizer(
            text_target=targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized = dataset_dict.map(
        preprocess,
        batched=True,
        remove_columns=["translation"]
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model
    )

    return tokenizer, model, tokenized, data_collator


def train_direction(
    dataset_dict,
    model_id,
    source_lang,
    target_lang,
    output_dir
):
    tokenizer, model, tokenized, data_collator = prepare_direction(
        dataset_dict, model_id, source_lang, target_lang
    )

    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        weight_decay=0.01,
        warmup_ratio=0.05,
        logging_steps=50,
        save_strategy="epoch",
        eval_strategy="epoch",
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
        load_best_model_at_end=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    print(f"Training {source_lang} -> {target_lang} using {model_id}")
    trainer.train()

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    return trainer, tokenizer, model


## 9. Fine-tune Urdu → English

This cell performs the required fine-tuning step for the Urdu-to-English model.

In [ ]:
# ============================================================
# 9A. Train Urdu -> English
# ============================================================
ur_en_train, ur_en_tokenizer, ur_en_model = train_direction(
    {"train": train_ds, "validation": valid_ds},
    UR_EN_BASE,
    source_lang="ur",
    target_lang="en",
    output_dir="/content/ur_en_finetuned"
)

print("Urdu -> English model saved to /content/ur_en_finetuned")


In [ ]:
# ============================================================
# 10. Evaluate Urdu -> English with BLEU
# ============================================================
def bleu_evaluate(model, tokenizer, test_data, source_lang, target_lang, max_items=None):
    if max_items:
        test_data = test_data.select(range(min(max_items, len(test_data))))

    sources = [x["translation"][source_lang] for x in test_data]
    references = [[x["translation"][target_lang]] for x in test_data]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    predictions = []
    for start in range(0, len(sources), 16):
        batch = sources[start:start+16]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4
            )

        predictions.extend(
            tokenizer.batch_decode(outputs, skip_special_tokens=True)
        )

    score = bleu_metric.compute(
        predictions=predictions,
        references=references
    )

    return score["score"], predictions, references

ur_en_bleu, ur_en_predictions, ur_en_references = bleu_evaluate(
    ur_en_model,
    ur_en_tokenizer,
    test_ds,
    "ur",
    "en"
)

print(f"Urdu -> English BLEU: {ur_en_bleu:.2f}")


## 11. Fine-tune English → Urdu

The same transfer-learning process is repeated in the opposite direction so the final application supports both directions.

In [ ]:
# ============================================================
# 11A. Train English -> Urdu
# ============================================================
en_ur_train, en_ur_tokenizer, en_ur_model = train_direction(
    {"train": train_ds, "validation": valid_ds},
    EN_UR_BASE,
    source_lang="en",
    target_lang="ur",
    output_dir="/content/en_ur_finetuned"
)

print("English -> Urdu model saved to /content/en_ur_finetuned")


In [ ]:
# ============================================================
# 12. Evaluate English -> Urdu with BLEU
# ============================================================
en_ur_bleu, en_ur_predictions, en_ur_references = bleu_evaluate(
    en_ur_model,
    en_ur_tokenizer,
    test_ds,
    "en",
    "ur"
)

print(f"English -> Urdu BLEU: {en_ur_bleu:.2f}")


In [ ]:
# ============================================================
# 13. Test both directions manually
# ============================================================
def translate(text, model, tokenizer):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

urdu_text = "پاکستان ایک خوبصورت ملک ہے۔"
english_text = "Pakistan is a beautiful country."

print("Urdu:", urdu_text)
print("English translation:", translate(urdu_text, ur_en_model, ur_en_tokenizer))

print()
print("English:", english_text)
print("Urdu translation:", translate(english_text, en_ur_model, en_ur_tokenizer))


## 14. Save the final models

The assignment requires the fine-tuned model to be saved and then uploaded to Hugging Face.

In [ ]:
# ============================================================
# 14. Verify saved model files
# ============================================================
from pathlib import Path

for folder in ["/content/ur_en_finetuned", "/content/en_ur_finetuned"]:
    print("\n", folder)
    for p in sorted(Path(folder).glob("*")):
        print(" -", p.name)


## 15. Upload the fine-tuned models to Hugging Face

Run the next cell and enter your Hugging Face token when prompted.

Create two repositories, for example:
- `YOUR_USERNAME/urdu-english-translation`
- `YOUR_USERNAME/english-urdu-translation`

The assignment specifically requires your own Hugging Face model repository.

In [ ]:
# ============================================================
# 15. Hugging Face login
# ============================================================
from huggingface_hub import notebook_login

notebook_login()
print("Logged in to Hugging Face.")


In [ ]:
# ============================================================
# 16. Push both fine-tuned models to your Hugging Face account
# ============================================================
# CHANGE THIS to your real Hugging Face username.
HF_USERNAME = "YOUR_HUGGINGFACE_USERNAME"

UR_EN_REPO = f"{HF_USERNAME}/urdu-english-translation"
EN_UR_REPO = f"{HF_USERNAME}/english-urdu-translation"

ur_en_model.push_to_hub(UR_EN_REPO)
ur_en_tokenizer.push_to_hub(UR_EN_REPO)

en_ur_model.push_to_hub(EN_UR_REPO)
en_ur_tokenizer.push_to_hub(EN_UR_REPO)

print("Uploaded both models.")
print("Urdu -> English:", f"https://huggingface.co/{UR_EN_REPO}")
print("English -> Urdu:", f"https://huggingface.co/{EN_UR_REPO}")


## 17. Generate the Streamlit application files

The assignment requires a user-friendly Streamlit application, GitHub source code, and connection of the GitHub project to Streamlit Community Cloud.

The next cell creates:
- `app.py`
- `requirements.txt`
- `README.md`

Then it packages them into a ZIP file that can be uploaded to GitHub.

In [ ]:
# ============================================================
# 17. Create the GitHub-ready Streamlit project
# ============================================================
from pathlib import Path
import zipfile

HF_USERNAME = "YOUR_HUGGINGFACE_USERNAME"   # <-- same username as step 16

project_dir = Path("/content/urdu_english_translation_streamlit")
FILES = {'app.py': '"""Urdu <-> English Neural Machine Translation - Streamlit app."""\nimport os\n\nimport streamlit as st\nimport torch\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer\n\nst.set_page_config(page_title="Urdu ↔ English Translator", page_icon="🌐", layout="centered")\n\n# ---------------------------------------------------------------------------\n# Configuration\n# Set HF_USERNAME in Streamlit Cloud > App settings > Secrets:\n#   HF_USERNAME = "your-huggingface-username"\n# If your fine-tuned repos are not found, the app falls back to the\n# pretrained Helsinki-NLP base models so it never crashes.\n# ---------------------------------------------------------------------------\ndef get_username() -> str:\n    try:\n        if "HF_USERNAME" in st.secrets:\n            return st.secrets["HF_USERNAME"]\n    except Exception:\n        pass\n    return os.environ.get("HF_USERNAME", "YOUR_HUGGINGFACE_USERNAME")\n\n\nHF_USERNAME = get_username()\n\nMODELS = {\n    "Urdu → English": {\n        "finetuned": f"{HF_USERNAME}/urdu-english-translation",\n        "base": "Helsinki-NLP/opus-mt-ur-en",\n    },\n    "English → Urdu": {\n        "finetuned": f"{HF_USERNAME}/english-urdu-translation",\n        "base": "Helsinki-NLP/opus-mt-en-ur",\n    },\n}\n\nEXAMPLES = {\n    "Urdu → English": ["پاکستان ایک خوبصورت ملک ہے۔", "آپ کا نام کیا ہے؟", "میں کل لاہور جا رہا ہوں۔"],\n    "English → Urdu": ["Pakistan is a beautiful country.", "What is your name?", "I am going to Lahore tomorrow."],\n}\n\n\n@st.cache_resource(show_spinner=False)\ndef load_model(direction: str):\n    """Load the fine-tuned model; fall back to the pretrained base model."""\n    cfg = MODELS[direction]\n    for repo, label in ((cfg["finetuned"], "fine-tuned"), (cfg["base"], "base")):\n        try:\n            tok = AutoTokenizer.from_pretrained(repo)\n            mdl = AutoModelForSeq2SeqLM.from_pretrained(repo)\n            mdl.eval()\n            return tok, mdl, repo, label\n        except Exception:\n            continue\n    raise RuntimeError("Could not load any model.")\n\n\ndef translate(text, tok, mdl, num_beams=4):\n    inputs = tok(text, return_tensors="pt", truncation=True, max_length=128)\n    with torch.no_grad():\n        out = mdl.generate(**inputs, max_length=128, num_beams=num_beams)\n    return tok.decode(out[0], skip_special_tokens=True)\n\n\n# ------------------------------- UI ----------------------------------------\nst.markdown(\n    """\n    <style>\n    textarea[aria-label="Result"] {font-size: 1.1rem;}\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\nst.title("🌐 Urdu ↔ English Translator")\nst.caption("Fine-tuned Hugging Face Marian models • trained on the OPUS-100 en-ur parallel corpus")\n\nwith st.sidebar:\n    st.header("Settings")\n    num_beams = st.slider("Beam size (quality vs speed)", 1, 8, 4)\n    st.markdown("**Dataset:** [Helsinki-NLP/opus-100 (en-ur)](https://huggingface.co/datasets/Helsinki-NLP/opus-100)")\n    st.markdown("**Base models:** [ur-en](https://huggingface.co/Helsinki-NLP/opus-mt-ur-en), [en-ur](https://huggingface.co/Helsinki-NLP/opus-mt-en-ur)")\n\ndirection = st.radio("Translation direction", list(MODELS.keys()), horizontal=True)\n\nexample = st.selectbox("Try an example", [""] + EXAMPLES[direction])\ntext = st.text_area("Enter text", value=example, height=150, placeholder="Type your sentence here...", max_chars=1000)\n\nif st.button("Translate", type="primary"):\n    if not text.strip():\n        st.warning("Please enter some text.")\n    else:\n        with st.spinner("Loading model and translating (first run may take a minute)..."):\n            tok, mdl, repo, label = load_model(direction)\n            result = translate(text.strip(), tok, mdl, num_beams)\n        st.success("Translation")\n        if direction == "English → Urdu":\n            st.markdown(f"<div dir=\'rtl\' style=\'font-size:1.4rem\'>{result}</div>", unsafe_allow_html=True)\n        else:\n            st.write(result)\n        st.code(result, language=None)\n        st.caption(f"Model used: `{repo}` ({label})")\n', 'requirements.txt': '--extra-index-url https://download.pytorch.org/whl/cpu\nstreamlit\ntorch\ntransformers\nsentencepiece\nsacremoses\nhuggingface_hub\n', 'README.md': '# Urdu ↔ English Neural Machine Translation\n\nEnd-to-end NLP project: fine-tune pretrained Hugging Face translation models on an online parallel corpus and deploy them with Streamlit.\n\n## Problem statement\nMany users need quick translation between Urdu and English. This project fine-tunes pretrained sequence-to-sequence models for both directions and serves them through a simple web app.\n\n## Dataset (online)\n[`Helsinki-NLP/opus-100`](https://huggingface.co/datasets/Helsinki-NLP/opus-100), configuration `en-ur` — downloaded automatically in the notebook with `load_dataset("Helsinki-NLP/opus-100", "en-ur")`. No local file is needed.\n\n## Pretrained models\n- Urdu → English: [`Helsinki-NLP/opus-mt-ur-en`](https://huggingface.co/Helsinki-NLP/opus-mt-ur-en)\n- English → Urdu: [`Helsinki-NLP/opus-mt-en-ur`](https://huggingface.co/Helsinki-NLP/opus-mt-en-ur)\n\n## Fine-tuned models (my Hugging Face repos)\n- `YOUR_HUGGINGFACE_USERNAME/urdu-english-translation`\n- `YOUR_HUGGINGFACE_USERNAME/english-urdu-translation`\n\n## Pipeline\n1. Load + clean OPUS-100 (en-ur) → 2. Tokenize → 3. Fine-tune both directions (`Seq2SeqTrainer`) → 4. Evaluate with BLEU (sacreBLEU) → 5. Push to Hugging Face → 6. Streamlit app.\n\nThe full workflow is in `Urdu_English_NLP_Translation_Project.ipynb` (run on Google Colab with a T4 GPU).\n\n## Run locally\n```bash\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n## Deploy on Streamlit Community Cloud\n1. Push this folder to a GitHub repository.\n2. Go to https://share.streamlit.io → **New app** → select your repo, branch `main`, file `app.py`.\n3. In **Advanced settings → Secrets** add:\n   ```toml\n   HF_USERNAME = "your-huggingface-username"\n   ```\n4. Click **Deploy**. If your fine-tuned repos are not found, the app automatically uses the pretrained base models.\n\n## Results\nAdd your BLEU scores here after running the notebook:\n- Urdu → English BLEU: _fill in_\n- English → Urdu BLEU: _fill in_\n', '.gitignore': '__pycache__/\n*.pyc\n.env\n.streamlit/secrets.toml\n*_finetuned/\n.ipynb_checkpoints/\n', '.streamlit/config.toml': '[theme]\nprimaryColor = "#1a7f64"\n\n[server]\nmaxUploadSize = 5\n'}

for name, content in FILES.items():
    content = content.replace("YOUR_HUGGINGFACE_USERNAME", HF_USERNAME)
    path = project_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

# also include this notebook's BLEU results in the README
try:
    with open(project_dir / "README.md", "a", encoding="utf-8") as f:
        f.write(f"\n\n**Measured BLEU** - ur->en: {ur_en_bleu:.2f}, en->ur: {en_ur_bleu:.2f}\n")
except NameError:
    pass

zip_path = Path("/content/urdu_english_translation_streamlit.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in project_dir.rglob("*"):
        if file.is_file():
            z.write(file, arcname=str(file.relative_to(project_dir)))

print("Created:", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    pass


## 18. Final checklist for submission

Before submitting, make sure you have:

1. **Project title + problem statement**
2. **Dataset source + description**
3. **Pretrained model names + Hugging Face links**
4. **Your own Hugging Face model repository links**
5. **Your GitHub repository link**
6. **Your live Streamlit application link**
7. **Brief report explaining the complete development process**

These items match the assignment's final submission requirements.

### Recommended project flow

**Colab**
→ download online OPUS-100 dataset  
→ clean/preprocess  
→ load pretrained models  
→ fine-tune both directions  
→ evaluate with BLEU  
→ save models  
→ push models to Hugging Face  
→ generate Streamlit/GitHub files

**GitHub**
→ upload `app.py`, `requirements.txt`, `README.md`

**Streamlit Community Cloud**
→ connect GitHub repository  
→ select `app.py`  
→ deploy  
→ test public URL



## Notes

- A **GPU runtime is strongly recommended** for the fine-tuning cells.
- The notebook intentionally uses a smaller subset so it is practical for a student Colab project. Increase `MAX_TRAIN_SAMPLES` if you have enough runtime.
- The dataset is downloaded online; no local dataset file is required.
- The two translation directions are fine-tuned separately because the selected Helsinki-NLP models are direction-specific. The Urdu→English model is explicitly documented as Urdu source / English target, and the English→Urdu model as English source / Urdu target.
- The assignment says students should understand every phase rather than simply copy an existing project.


## 19. Push to GitHub and deploy on Streamlit
1. Unzip `urdu_english_translation_streamlit.zip`, and put this notebook in the same folder.
2. `git init && git add . && git commit -m "Urdu-English translator"`, then create a GitHub repo and `git push`.
3. Open https://share.streamlit.io -> New app -> choose your repo, branch `main`, file `app.py`.
4. In Advanced settings -> Secrets add `HF_USERNAME = "your-username"`, then Deploy.
